# Data-to-text controllato — laboratorio guidato

**Dal profilo di un'amministrazione a una spiegazione verificabile.**

Questo notebook affianca l'applicazione: puoi leggere e modificare regole, template e feature, osservando ogni passaggio. Non servono MongoDB o servizi di generazione testuale.

## Percorso
1. Prepara il progetto e carica il pacchetto.
2. Scegli tipologia e amministrazione.
3. Esplora indicatori e serie annuali.
4. Seleziona i contenuti con regole esplicite.
5. Personalizza i template e verifica il determinismo.
6. Estrai le feature osservabili e controlla i mancanti.
7. Facoltativamente, addestra il comitato sul laboratorio.
8. Esporta testo, evidenze e configurazione.

**Prima di iniziare:** dall'app scarica il “Pacchetto per Colab”, che contiene codice e profilo selezionato. Il notebook non carica automaticamente dati su servizi esterni: in Colab scegli tu il file da caricare. Un pacchetto con un solo profilo avrà un solo ente nei selettori; lo ZIP completo del progetto permette di esplorare l'intera copia locale.

## 1 · Preparazione

In Colab scegli un runtime CPU ed esegui le celle in ordine. Se il progetto non è presente, verrà richiesto lo ZIP del pacchetto. Non sono necessari token GitHub: il caricamento manuale funziona anche con un repository privato.

In locale apri il notebook dal progetto e installa prima `python -m pip install -e ".[notebook]"`. La cella seguente installa dipendenze solo quando viene eseguita in Colab.

In [ ]:
from pathlib import Path
import sys, json, io, zipfile, subprocess
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# In locale puoi impostare qui un percorso esplicito.
PROJECT_DIR = ""
candidates = ([Path(PROJECT_DIR)] if PROJECT_DIR else []) + [Path.cwd(), Path.cwd().parent]
ROOT = next((p.resolve() for p in candidates if (p / "src/controlled_d2t").is_dir()), None)
if ROOT is None:
    if not IN_COLAB:
        raise RuntimeError("Imposta PROJECT_DIR o apri il notebook nella cartella del progetto.")
    print("Carica il Pacchetto per Colab oppure lo ZIP completo del progetto.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Caricare un solo archivio ZIP.")
    destination = Path("/content/d2t_workshop").resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
        if sum(info.file_size for info in archive.infolist()) > 300_000_000:
            raise ValueError("Archivio troppo grande per questo laboratorio.")
        for info in archive.infolist():
            target = (destination / info.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Percorso non valido nell'archivio.")
            if (info.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("I collegamenti simbolici non sono ammessi.")
        archive.extractall(destination)
    projects = [p.parent for p in destination.rglob("pyproject.toml") if (p.parent / "src/controlled_d2t").is_dir()]
    if len(projects) != 1:
        raise ValueError("Impossibile individuare un unico progetto nel pacchetto.")
    ROOT = projects[0]
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.26,<3", "pandas>=2.2,<3", "scikit-learn>=1.5,<2", "ipywidgets>=8,<9", "matplotlib>=3.8,<4"])
    # Il notebook usa le API Python; non avvia l'interfaccia Streamlit.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
sys.path.insert(0, str(ROOT / "src"))
print("Progetto pronto:", ROOT)

## 2 · Dati e selezione

Il pacchetto dedicato conserva il profilo scelto nell'app. Se usi lo ZIP completo vengono letti i profili reali disponibili. In assenza di copie esportate viene creato un piccolo laboratorio esplicitamente identificato.

La funzione di raggruppamento è la stessa dell'applicazione: prima la tipologia, poi l'ente. I nomi dei campi restano visibili, così puoi verificare il contratto dei dati.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from controlled_d2t.selection import group_profiles, typology
from controlled_d2t.synthetic import load, make_dataset
from controlled_d2t.schema import validate, BLOCKS, FEATURES
from controlled_d2t.engine import generate, DEFAULT_TEMPLATES
from controlled_d2t.features import extract_observed_features

paths = [ROOT / "data/colab/profiles.jsonl", ROOT / "data/real/profiles.jsonl", ROOT / "data/synthetic.jsonl"]
DATA_PATH = next((path for path in paths if path.exists()), None)
profiles = load(DATA_PATH) if DATA_PATH else make_dataset(100, seed=42)
groups = group_profiles(profiles)
selected_type = max(groups, key=lambda kind: len(groups[kind]))
selected_index = groups[selected_type][0]
profile = profiles[selected_index]
display(pd.DataFrame([{"Tipologia": key, "Enti disponibili": len(value)} for key, value in groups.items()]))
print("Fonte:", DATA_PATH or "Laboratorio generato localmente")
print("Ente:", profile["denominazione"], "| Codice:", profile["codice_ente"])
print("Profilo dimostrativo:", profile.get("synthetic") is True)

### Selettori interattivi
Puoi cambiare l'ente qui e rieseguire le celle successive. La selezione aggiorna la variabile `profile`; i risultati già calcolati nelle celle precedenti non vengono riscritti automaticamente.

In [ ]:
import ipywidgets as widgets
type_widget = widgets.Dropdown(options=sorted(groups), value=selected_type, description="Tipologia:", layout=widgets.Layout(width="95%"))
entity_widget = widgets.Dropdown(options=[(profiles[i]["denominazione"], i) for i in groups[selected_type]], value=selected_index, description="Ente:", layout=widgets.Layout(width="95%"))
selection_output = widgets.Output()

def update_entity(change=None):
    global profile
    profile = profiles[entity_widget.value]
    with selection_output:
        selection_output.clear_output(wait=True)
        print(profile["codice_ente"], profile["denominazione"])

def update_type(change):
    entity_widget.options = [(profiles[i]["denominazione"], i) for i in groups[change["new"]]]
    entity_widget.value = groups[change["new"]][0]
    update_entity()

type_widget.observe(update_type, names="value")
entity_widget.observe(update_entity, names="value")
display(widgets.VBox([type_widget, entity_widget, selection_output]))
update_entity()

## 3 · Leggi e controlla il profilo

La validazione verifica valori finiti, scale esplicite e coerenza tra rapporti e flussi annuali. Non certifica la fonte né la correttezza di tutte le formule economiche a monte. Un rapporto negativo indica rientro netto; non è una misura dello stock di debito.

In [ ]:
errors = validate(profile)
if errors:
    raise ValueError("Profilo non valido: " + "; ".join(errors))
keys = ["entrate_medie", "popolazione_media", "indice_medio_debito_su_entrate", "indice_efficienza", "indice_sostenibilita", "indice_rischio"]
display(pd.DataFrame({"Campo": keys, "Valore": [profile.get(key) for key in keys]}))
annual = pd.DataFrame.from_dict(profile.get("valori_annuali", {}), orient="index")
if not annual.empty:
    annual.index = annual.index.astype(int)
    annual = annual.sort_index()
    display(annual)
    ratio_series = pd.to_numeric(annual["debito_su_entrate"], errors="coerce") * 100
    ratio_series = ratio_series.reindex(range(annual.index.min(), annual.index.max()+1))
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(ratio_series.index, ratio_series.values, color="#087e83", marker="o")
    ax.axhline(0, color="#647988", linestyle="--", linewidth=1)
    ax.set(xlabel="Anno", ylabel="Flusso netto / entrate (%)", title="Rapporto annuale: nessuna interpolazione degli anni mancanti")
    ax.grid(alpha=.15)
    display(fig)
    plt.close(fig)

## 4 · Seleziona i blocchi narrativi

Separiamo due decisioni: **il contenuto richiesto** e **la disponibilità delle evidenze**. La selezione manuale non può forzare un contenuto privo dei dati necessari.

Modifica `requested_blocks` e riesegui la cella. Confronta la spiegazione completa con quella ridotta: quali informazioni hai escluso?

In [ ]:
full_result = generate(profile)
display(pd.DataFrame(full_result["decisions"]))
requested_blocks = {"introduzione", "entrate", "debito", "trend"}
selected = {block: block in requested_blocks for block in BLOCKS}
reduced_result = generate(profile, selected=selected)
display(Markdown("### Spiegazione con i contenuti selezionati"))
display(Markdown(reduced_result["text"]))
display(pd.DataFrame(reduced_result["decisions"]))

### Esperimento: rimuovere un dato

La copia di prova non modifica il profilo originale. Anche richiedendo il blocco debito, il motore deve escluderlo quando manca il rapporto.

In [ ]:
from copy import deepcopy
missing_profile = deepcopy(profile)
missing_profile["indice_medio_debito_su_entrate"] = None
missing_result = generate(missing_profile, selected={block: block == "debito" for block in BLOCKS})
assert not any(block["id"] == "debito" for block in missing_result["blocks"])
print("Controllo superato: nessuna frase sul rapporto mancante.")

## 5 · Template, segnaposto e determinismo

I template sono frasi vincolate con segnaposto espliciti. La scelta della variante usa un hash stabile di codice ente e blocco. Modificare la denominazione cambia il dato inserito, non introduce una nuova scelta casuale.

Puoi modificare i testi della cella successiva mantenendo `{nome}` e gli altri segnaposto disponibili. Non aggiungere affermazioni che i dati non sostengono.

In [ ]:
from string import Formatter
templates = json.loads(DEFAULT_TEMPLATES.read_text(encoding="utf-8"))
catalog = []
for block, variants in templates.items():
    placeholders = sorted({name for text in variants for _, name, _, _ in Formatter().parse(text) if name})
    catalog.append({"Blocco": block, "Varianti": len(variants), "Segnaposto": ", ".join(placeholders)})
display(pd.DataFrame(catalog))
print(json.dumps({"introduzione": templates["introduzione"], "debito": templates["debito"]}, ensure_ascii=False, indent=2))

In [ ]:
OUTPUT = ROOT / "work/colab"
OUTPUT.mkdir(parents=True, exist_ok=True)
custom_templates = deepcopy(templates)
custom_templates["introduzione"] = [
    "Questa scheda presenta i dati disponibili per {nome}.",
    "Il profilo di {nome} viene descritto attraverso indicatori verificabili.",
]
custom_path = OUTPUT / "template_personalizzati.json"
custom_path.write_text(json.dumps(custom_templates, ensure_ascii=False, indent=2), encoding="utf-8")
custom_result = generate(profile, template_path=custom_path)
assert custom_result["status"] != "blocked", custom_result["errors"]
assert custom_result == generate(deepcopy(profile), template_path=custom_path)
display(Markdown(custom_result["text"]))
print("Stesso input e stessi template: output identico.")
display(pd.DataFrame([{key: block[key] for key in ["id", "template_id", "rule_id"]} for block in custom_result["blocks"]]))

## 6 · Feature osservate: numeri, unità e mancanti

L'estrattore produce **18 feature osservabili**: 7 strutturali, 3 indici e 8 variazioni annuali. Non riempie lo schema dimostrativo con valori inventati.

- `pop_log` e `entrate_log`: log naturale di 1 + valore; null se manca.
- `personale_presente` e `appalti_presenti`: presenza del dato, anche se zero.
- `quota_mancanti`: quota sui quattro campi dichiarati nel codice; non è un indice generale di qualità.
- `trend_debito_pp_1`: differenza tra ultimo anno e precedente in punti percentuali. Gli altri confronti risalgono nel tempo; un anno mancante produce null.

Il modello dimostrativo richiede invece 42 colonne con definizioni diverse. **Non passare queste 18 feature al modello a 42 colonne.**

In [ ]:
observed = extract_observed_features(profile)
feature_table = pd.DataFrame([{"Feature": key, "Valore": value, "Disponibile": value is not None} for key, value in observed.items()])
display(feature_table)
assert len(observed) == 18
print("Feature disponibili:", sum(value is not None for value in observed.values()), "/", len(observed))
import inspect
print(inspect.getsource(extract_observed_features))

### Esperimento: zero non significa dato mancante

In [ ]:
zero_profile = deepcopy(profile)
zero_profile["indice_efficienza"] = 0
zero_profile["PERSONALE"] = 0
zero_profile["popolazione_media"] = None
zero_features = extract_observed_features(zero_profile)
assert zero_features["score_efficienza"] == 0
assert zero_features["personale_presente"] == 1
assert zero_features["pop_log"] is None
print({key: zero_features[key] for key in ["score_efficienza", "personale_presente", "pop_log"]})

## 7 · Laboratorio ML opzionale

Questa parte crea profili ed etichette tramite una policy simulata e addestra modelli nuovi nel runtime corrente. Non usa le spiegazioni reali come annotazioni e non replica i risultati del paper. Non carica file pickle ricevuti dall'esterno.

Il confronto misura soprattutto la capacità di apprendere la policy e i limiti della rappresentazione. Il margine del modello non è una probabilità calibrata di correttezza. Imposta `TRAIN_MODELS = True` per eseguire l'addestramento; aumenta `N_LAB` e `BOOTSTRAP` per un esperimento più ampio.

In [ ]:
from controlled_d2t.experiment import matrix
lab_profiles = make_dataset(300, seed=42)
X = matrix(lab_profiles)
assert X.shape == (300, 42)
display(pd.DataFrame({"Colonna": range(42), "Feature": FEATURES, "Primo profilo": X[0]}))
print("Matrice del laboratorio:", X.shape)

In [ ]:
TRAIN_MODELS = False
N_LAB = 300
BOOTSTRAP = 200
lab_report = None
if TRAIN_MODELS:
    from controlled_d2t.experiment import run
    lab_report = run(make_dataset(N_LAB, seed=42), OUTPUT / "experiment", seed=42, bootstrap=BOOTSTRAP)
    display(pd.DataFrame(lab_report["metrics"]).T)
    print(lab_report["warning"])
else:
    print("Addestramento non eseguito. Imposta TRAIN_MODELS = True per attivarlo.")

## 8 · Esporta risultati e provenienza

Il rapporto salva input, feature osservate, template, decisioni, evidenze e versioni. Il testo rimane una bozza da revisionare. Non vengono pubblicati risultati né scritti dati su MongoDB.

In [ ]:
import hashlib, platform
current_input_hash = hashlib.sha256(json.dumps(profile, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()
if custom_result.get("input_sha256") != current_input_hash:
    raise RuntimeError("Il profilo è cambiato: riesegui le celle da Template in poi prima di esportare.")
report = {
    "profile": profile,
    "observed_features": extract_observed_features(profile),
    "generation": custom_result,
    "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__},
    "input_sha256": hashlib.sha256(json.dumps(profile, sort_keys=True, ensure_ascii=False).encode()).hexdigest(),
    "laboratory_metrics": lab_report,
}
(OUTPUT / "rapporto.json").write_text(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
(OUTPUT / "spiegazione.txt").write_text(custom_result["text"], encoding="utf-8")
export_zip = OUTPUT / "risultati-notebook.zip"
with zipfile.ZipFile(export_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ["rapporto.json", "spiegazione.txt", "template_personalizzati.json"]:
        archive.write(OUTPUT / name, arcname=name)
print("Risultati pronti:", export_zip)

In [ ]:
# In Colab attiva questa opzione per scaricare il rapporto sul tuo computer.
DOWNLOAD_RESULTS = False
if IN_COLAB and DOWNLOAD_RESULTS:
    files.download(str(export_zip))

## Estensioni proposte

1. Confronta due versioni dei template senza cambiare i numeri.
2. Definisci una policy esplicita di selezione dei blocchi e giustifica le esclusioni.
3. Analizza le feature mancanti per tipologia senza confondere mancanti e non applicabilità.
4. Raccogli annotazioni indipendenti prima di addestrare e valutare un modello sui profili reali.

### Riferimenti
- [Paper di riferimento](https://claudiobiancalana.it/assets/publications/biancalana-sbicca-ital-ia-2026.pdf).
- [Guida ufficiale Colab: caricamento di notebook e file](https://research.google.com/colaboratory/intl/en-GB/faq.html).

Il notebook è distribuito senza output eseguiti o dati incorporati. Colab esegue il codice nel tuo runtime; il suo `localhost` non è il computer su cui si trova MongoDB. Per questo si lavora su copie di file.